# How do daily stats effect sleep?


## Overview of the daily health stats

In [6]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.utils.feature_engineering import add_sleep_features
from scipy.stats import pearsonr


In [4]:
# The data
con = sqlite3.connect(r"C:\Users\Dexta\Learning\Garmin-Analysis\data\garmin.db")

dailySummary_raw = pd.read_sql(
    "SELECT * FROM DailySummary",
    con
)

sleep_raw = pd.read_sql(
    "SELECT * FROM Sleep",
    con
)

In [5]:
dailySummary_raw.columns


Index(['calendarDate', 'activeKilocalories', 'bmrKilocalories', 'totalSteps',
       'totalDistanceMeters', 'highlyActiveSeconds', 'activeSeconds',
       'moderateIntensityMinutes', 'vigorousIntensityMinutes', 'minHeartRate',
       'maxHeartRate', 'currentDayRestingHeartRate', 'minAvgHeartRate',
       'maxAvgHeartRate', 'averageSpo2Value', 'lowestSpo2Value',
       'sweatLossInML', 'highestRespirationValue', 'lowestRespirationValue',
       'avgRespirationValue', 'charged', 'drained', 'endBattery', 'minBattery',
       'maxBattery', 'averageStressLevel', 'maxStressLevel',
       'lowStressDuration', 'mediumStressDuration', 'highStressDuration'],
      dtype='object')

In [7]:

# Align sleep + daily on calendarDate
daily = dailySummary_raw.copy()
sleep = add_sleep_features(sleep_raw)

daily["calendarDate"] = pd.to_datetime(daily["calendarDate"])
sleep["calendarDate"] = pd.to_datetime(sleep["calendarDate"])

merged = daily.merge(sleep, on="calendarDate", how="inner")

# Numeric daily features
daily_cols = merged[dailySummary_raw.select_dtypes(include=[np.number]).columns].columns

# Numeric sleep features
sleep_cols = merged[add_sleep_features(sleep_raw).select_dtypes(include=[np.number]).columns].columns

results = []

for s_col in sleep_cols:
    for d_col in daily_cols:

        x = merged[s_col].values.astype(float)
        y = merged[d_col].values.astype(float)

        # Remove NaN/inf pairs
        mask = np.isfinite(x) & np.isfinite(y)
        x_clean = x[mask]
        y_clean = y[mask]

        # Need at least 3 points
        if len(x_clean) < 3:
            continue

        # Skip constant arrays
        if np.nanstd(x_clean) == 0 or np.nanstd(y_clean) == 0:
            continue

        # Pearson correlation
        r, p = pearsonr(x_clean, y_clean)

        if p < 0.05:
            results.append((s_col, d_col, r, p))

# Print results
if not results:
    print("No significant sleep → daily correlations found.")
else:
    print("Significant Sleep → Daily Correlations (p < 0.05):\n")
    for s_col, d_col, r, p in sorted(results, key=lambda x: abs(x[2]), reverse=True):
        print(f"{s_col}  →  {d_col}   r={r:.3f}, p={p:.4f}")


Significant Sleep → Daily Correlations (p < 0.05):

lowestSPO2  →  lowestSpo2Value   r=0.908, p=0.0000
averageSPO2  →  averageSpo2Value   r=0.871, p=0.0000
avgSleepStress  →  maxBattery   r=-0.756, p=0.0000
averageHR  →  currentDayRestingHeartRate   r=0.741, p=0.0000
averageHR  →  maxBattery   r=-0.706, p=0.0000
recoveryScore  →  maxBattery   r=0.672, p=0.0000
avgSleepStress  →  drained   r=-0.626, p=0.0000
averageSPO2  →  lowestSpo2Value   r=0.603, p=0.0000
averageHR  →  minAvgHeartRate   r=0.594, p=0.0000
lowestSPO2  →  averageSpo2Value   r=0.591, p=0.0000
totalSleepSeconds  →  maxBattery   r=0.590, p=0.0000
sleep_hours  →  maxBattery   r=0.590, p=0.0000
centered_sleep_hours  →  maxBattery   r=0.590, p=0.0000
lowestRespiration  →  lowestRespirationValue   r=0.589, p=0.0000
durationScore  →  maxBattery   r=0.575, p=0.0000
overallScore  →  maxBattery   r=0.559, p=0.0000
averageHR  →  minHeartRate   r=0.557, p=0.0000
avgSleepStress  →  currentDayRestingHeartRate   r=0.556, p=0.0000
aver